CTP 2 - Exercise 1 STARTER: profile customers.csv with pandas.  
Run from the 02_CTP2 folder: `python profile_customers_starter.py`  
Goal: measure at least 8 rules (2 per dimension) and copy the numbers into quality_scorecard_template.csv 

In [64]:
import pandas as pd

df = pd.read_csv("../shopfast-kit/data/customers.csv", dtype=str)   # dtype=str: keep the raw text (e.g. 01300)

In [103]:
n = len(df)
print("Rows:", n)
print(df.head())

# --- Completeness: how many empty values per column? ---
print("\nEmpty values per column:\n", df.isna().sum())

# --- Uniqueness: is customer_id unique? ---
print("\nDuplicated customer_id rows:", df["customer_id"].duplicated(keep=False).sum())

Rows: 1241
  customer_id first_name last_name                         email  \
0       10328      Alain    Boutin        alain.boutin@gmail.com   
1       10524    Thérèse    Hubert    therese.hubert@outlook.com   
2       10125     Célina   Lemaire         celina.lemaire@gmx.de   
3       10812    Aurelia    Cuevas       aurelia.cuevas@yahoo.fr   
4       10335   Cristina    Campoy  cristina.campoy@@outlook.com   

               phone  birth_date                               address  \
0  +33 6 28 32 68 42  1956-08-31                    29, rue de Rolland   
1     06.32.83.38.98  1999-01-23                      875, rue Thierry   
2     06.65.40.51.53  09-04-1986           58, avenue Charlotte Berger   
3       +34666395517  1987-01-23            Cuesta de Jacinto Querol 6   
4       +34634110809  2006-01-10  Cuesta de Constanza Pla 98 Puerta 9    

                 city postal_code country signup_date marketing_consent  \
0             Jacquot       63965      FR  2023-08-29       

TODO 1 (Uniqueness): normalise email (strip + lower case) and count people registered twice  


In [66]:
email_norm = df["email"].str.strip().str.lower()

doublons_count = email_norm.duplicated(keep=False).sum()

print("Duplicated email:", doublons_count)

Duplicated email: 168


TODO 2 (Validity): count emails that do not match the regex  `^[^@\s]+@[^@\s]+\.[a-zA-Z]{2,}$`   (hint: .str.match)  


In [67]:
email_pattern = r"^[^@\s]+@[^@\s]+\.[a-zA-Z]{2,}$"
email_valide = email_norm.str.match(email_pattern, na=False)
emails_invalides_count = (~email_valide).sum()

print("Invalid emails:", emails_invalides_count)

Invalid emails: 67


TODO 3 (Consistency): list the distinct values of country with value_counts() - how many spellings for 4 countries?  


In [68]:
print(df["country"].value_counts())

unique_spellings_count = df["country"].nunique()
print("\nUnique spellings:", unique_spellings_count)

country
FR           530
DE           145
BE           127
ES           103
France        68
fr            53
france        41
FRA           31
Spain         14
DEU           14
Allemagne     13
Germany       12
Espagne       11
Belgium       11
BEL           11
es            11
Belgique       9
Name: count, dtype: int64

Unique spellings: 17


TODO 4 (Consistency): count signup_date values that are NOT in the format YYYY-MM-DD  


In [69]:
format_date = r"\d{4}-\d{2}-\d{2}"
date_invalide = ~df["signup_date"].str.contains(format_date)

invalid_dates_count = date_invalide.sum()
print("Signup dates not in YYYY-MM-DD format:", invalid_dates_count)

Signup dates not in YYYY-MM-DD format: 303


TODO 5 (Validity): loyalty_tier values outside bronze / silver / gold / platinum  


In [70]:
tiers_autorises = {"bronze", "silver", "gold", "platinum"}

tier_invalide = ~df["loyalty_tier"].isin(tiers_autorises)
tier_invalide_count = tier_invalide.sum()
print("Invalid loyalty tiers:", tier_invalide_count)

Invalid loyalty tiers: 13


TODO 6 (Accuracy): birth dates in the future or before 1906 (convert the ISO ones with pd.to_datetime)  


In [71]:
birth_dates = pd.to_datetime(
    df["birth_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

aujourd_hui = pd.Timestamp.today()
date_before_1906 = birth_dates < pd.Timestamp("1906-01-01")
date_in_future = birth_dates > aujourd_hui

print("Birth dates in the future:", date_in_future.sum())
print("Birth dates before 1906:", date_before_1906.sum())

invalid_birth_dates_count = date_in_future.sum() + date_before_1906.sum()

Birth dates in the future: 5
Birth dates before 1906: 3


TODO 7 (Validity): French postal codes that do not have 5 digits  


In [95]:
pays_norm = df["country"].str.strip().str.upper()
est_france = pays_norm.isin({"FR","FRA", "FRANCE"})

code_postal_valide = df["postal_code"].str.contains(r"\d{5}")

postal_francais_invalide = est_france & ~code_postal_valide

postal_francais_invalide_count = postal_francais_invalide.sum()
print("Invalid French postal codes:", postal_francais_invalide_count)

Invalid French postal codes: 48


Bonus: compute score_pct = 100 * (1 - failing_rows / n) for each rule  

In [ ]:
failing_rows = {
    "Duplicated customer_id": df["customer_id"].duplicated(keep=False).sum(),
    "Duplicated email": doublons_count,
    "Invalid emails": emails_invalides_count,
    "Signup dates not in YYYY-MM-DD format": invalid_dates_count,
    "Invalid loyalty tiers": tier_invalide_count,
    "Birth dates in the future or before 1906": invalid_birth_dates_count,
    "Invalid French postal codes": postal_francais_invalide_count
}

score_pct = pd.Series({
    rule: 100 * (1 - count / n)
    for rule, count in failing_rows.items()
})

print(score_pct.round(2))

Duplicated customer_id                      98.71
Duplicated email                            86.46
Invalid emails                              94.60
Signup dates not in YYYY-MM-DD format       75.58
Invalid loyalty tiers                       98.95
Birth dates in the future or before 1906    99.36
Invalid French postal codes                 96.13
dtype: float64
